<a href="https://colab.research.google.com/github/RoselindSi/uapp/blob/main/notebooks/colab_saprot_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UAPP — Structure-aware backbone (SaProt) on T2837 + S669

**Question.** §11 showed that ESM2-650M's σ-branch ranking transferred from T2837 to S669 (Spearman 0.348 → 0.434), but μ-accuracy did not (RMSE 1.50 → 2.83).  The campaign so far swapped datasets twice but kept the encoder fixed.  Can a structure-aware backbone (**SaProt** = AA + 3Di tokens via FoldSeek) rescue μ on the new label distribution while preserving σ-ranking?

**Method.** Drop-in encoder swap.  We rebuild the embedding cache with SaProt for both T2837 and S669, keeping the rest of the pipeline (D5 head, ensemble training, σ recalibration) identical to §11.

**Why both datasets, not just S669.**
1. T2837 is the in-distribution baseline — without it we can't tell whether SaProt is genuinely improving or just shifting the basin.
2. The strict-improvement criterion needs both halves to hold:
   - T2837: Spearman ≥ 0.348 and RMSE ≤ 1.50  (don't break what works)
   - S669:  RMSE significantly < 2.83  (the actual hypothesis)
3. The σ recalibration scalar (T = 2.76 with ESM2) is encoder-specific; it must be re-fitted for SaProt on S669.

**Compute.** ESM2-650M took ~3 min on T4 to cache T2837 (2584 mutations across 108 proteins).  SaProt is the same parameter count and hidden size; expect the same wall-clock plus ~1 min for FoldSeek.  Total ~10 min for both datasets.

## 1. Environment

In [1]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

GPU 0: Tesla T4 (UUID: GPU-880b9754-767d-1582-ed79-fc0878ad63e6)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content
!rm -rf uapp
!git clone https://github.com/RoselindSi/uapp.git
%cd /content/uapp
!git checkout claude/saprot-backbone   # use this PR's branch until merged
!git log --oneline -3

/content
Cloning into 'uapp'...
remote: Enumerating objects: 318, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 318 (delta 38), reused 37 (delta 15), pack-reused 227 (from 1)
Receiving objects: 100% (318/318), 8.78 MiB | 18.77 MiB/s, done.
Resolving deltas: 100% (150/150), done.
/content/uapp
Branch 'claude/saprot-backbone' set up to track remote branch 'claude/saprot-backbone' from 'origin'.
Switched to a new branch 'claude/saprot-backbone'
40e9262 (HEAD -> claude/saprot-backbone, origin/claude/saprot-backbone) Fix AF PDB path: script 14 writes to cache/af_pdbs, not ./af_pdbs
4f7139e SaProt structure-aware backbone scaffolding (T2837 + S669 eval)
cfcff3d REPORT §11: S669 external validation + post-hoc σ recalibration (script 19)


In [3]:
%cd /content/uapp
!git checkout main
!git pull --ff-only
!git log --oneline -3   # 应该看到 PR #24 的 commit "Script 20: re-resolve mut_idx ..."

/content/uapp
Switched to branch 'main'
Your branch is up to date with 'origin/main'.
Already up to date.
52945ad (HEAD -> main, origin/main, origin/HEAD) Script 20: re-resolve mut_idx against PDB-AA so SaProt samples the right residue
47f0470 (origin/claude/saprot-mutidx-pdb-realign) Script 20: re-resolve mut_idx against PDB-derived AA, not metadata sequence
7c6810c Fix AF PDB path in SaProt notebook (cache/af_pdbs vs ./af_pdbs)


In [4]:
!pip install -q transformers torch numpy pandas tqdm scipy biopython scikit-learn

!wget -q https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz
!tar -xzf foldseek-linux-avx2.tar.gz
import os
os.environ["PATH"] = os.getcwd() + "/foldseek/bin:" + os.environ["PATH"]

!foldseek version

8dc75c74ad0eddab73cfd905963d13bf74dc012b


## 2. Restore prerequisites from Drive

We need:
- T2837 metadata + ESM2 cache (for the split assignment and the σ-recalibration baseline)
- S669 metadata + ESM2 cache + bundled WT PDBs (already saved to Drive in the §11 run)
- T2837 AlphaFold-DB PDBs from scripts/14 (one per uniprot_id)

If T2837 AF PDBs are not in Drive, the next cell rebuilds them via scripts/14 (~5 min).

In [ ]:
import os, shutil
os.makedirs('cache', exist_ok=True)
os.makedirs('data/s669/S669', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# ── single-file restores ──────────────────────────────────────────────────────
to_restore = [
    # ESM2 T2837 cache
    'cache/t2837_embeddings_v2_650m.pt',
    'cache/t2837_metadata.csv',
    'cache/t2837_bio_features_650m_extended.pt',
    # S669 ESM2 cache
    'cache/s669_metadata.csv',
    'cache/s669_metadata_processed.csv',
    'cache/s669_metadata_with_pdb.csv',        # needed by §11 ESM-IF S669
    'cache/s669_embeddings_650m.pt',
    'cache/s669_bio_features_650m_extended.pt',
    # SaProt T2837 cache (produced by §4 / cell 12)
    'cache/t2837_metadata_saprot.csv',          # needed by §10 script 21
    'cache/t2837_embeddings_saprot.pt',
    'cache/t2837_bio_features_saprot_extended.pt',
    # SaProt S669 cache
    'cache/s669_metadata_saprot.csv',
    'cache/s669_embeddings_saprot.pt',
    'cache/s669_bio_features_saprot_extended.pt',
]
for path in to_restore:
    src = f'/content/drive/MyDrive/uapp_cache/{os.path.basename(path)}'
    if os.path.exists(src):
        shutil.copy(src, path)
        print(f'✓ {path}')
    else:
        print(f'✗ missing: {path}')

# ── directory restores (outputs from previous sessions) ───────────────────────
for outdir in ('saprot_eval_d5', 'saprot_recalibration',
               'esm2_aligned_eval_d5',
               'esmif_eval_d5', 'esmif_recalibration'):
    src_dir = f'/content/drive/MyDrive/uapp_cache/{outdir}'
    dst_dir = f'outputs/{outdir}'
    if os.path.isdir(src_dir):
        shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
        print(f'✓ outputs/{outdir}')
    else:
        print(f'✗ missing: outputs/{outdir}  (will be created when that section runs)')

# ── S669 PDBs ─────────────────────────────────────────────────────────────────
if not os.path.exists('data/s669/S669/pdbs') or not os.listdir('data/s669/S669/pdbs'):
    print('\nFetching S669.zip from Zenodo (one-time)...')
    !wget -q -O data/s669/S669.zip https://zenodo.org/records/7568094/files/S669.zip
    !unzip -o -q data/s669/S669.zip -d data/s669/
print(f's669 pdbs: {len(os.listdir("data/s669/S669/pdbs"))}')


## 3. T2837 AlphaFold-DB PDBs

scripts/14 downloads one AF DB model per `uniprot_id` (~99 unique).  If the cache is in Drive, restore it; otherwise rebuild.

In [7]:
import os
drive_af = '/content/drive/MyDrive/uapp_cache/af_pdbs'
local_af = 'af_pdbs'
script14_af = 'cache/af_pdbs'  # where script 14 actually writes

if os.path.exists(drive_af) and os.listdir(drive_af):
    !cp -r {drive_af} {local_af}
    print(f'restored {len(os.listdir(local_af))} AF PDBs from Drive')
elif os.path.exists(script14_af) and os.listdir(script14_af):
    if not os.path.exists(local_af):
        os.symlink(os.path.abspath(script14_af), local_af)
    print(f'using existing {len(os.listdir(local_af))} AF PDBs at {script14_af}')
else:
    print('AF PDBs not in Drive — rebuilding via scripts/14 (one-time, ~5 min)...')
    !python scripts/14_compute_structural_features.py \
        --metadata-csv cache/t2837_metadata.csv \
        --embeddings   cache/t2837_embeddings_v2_650m.pt \
        --extended-bio cache/t2837_bio_features_650m_extended.pt \
        --out          /tmp/t2837_dssp_features.pt
    # Script 14 caches PDBs at cache/af_pdbs — symlink to ./af_pdbs for the rest of the notebook
    if os.path.exists(script14_af) and not os.path.exists(local_af):
        os.symlink(os.path.abspath(script14_af), local_af)
    print(f'cached {len(os.listdir(local_af))} AF PDBs')

# Mirror to Drive so future sessions don't re-download
!mkdir -p {drive_af}
!cp -n {local_af}/*.pdb {drive_af}/ 2>/dev/null
print(f"Drive mirror: {len(os.listdir(drive_af))} PDBs")

restored 92 AF PDBs from Drive
Drive mirror: 92 PDBs


## 4. Cache T2837 SaProt embeddings (~5 min on T4)

Same mutation-aware feature shape as ESM2 (`h_site + h_window + wt_oh + mut_oh`, 2600-d), so all downstream scripts (06, 18, 19) work unchanged.

**Crucial:** we use the *same split* as the ESM2 T2837 cache to keep an apples-to-apples comparison.  Script 20 inherits `split` from the metadata CSV when present.

In [8]:
!python scripts/20_cache_embeddings_saprot.py \
    --metadata-csv cache/t2837_metadata.csv \
    --pdb-dir      af_pdbs \
    --pdb-pattern  'AF-{uniprot_id}.pdb' \
    --out          cache/t2837_embeddings_saprot.pt \
    --metadata-out cache/t2837_metadata_saprot.csv \
    --device cuda --seed 42

19:36:06 [INFO] uapp: device: cuda
19:36:06 [INFO] uapp: loading metadata: cache/t2837_metadata.csv
19:36:06 [INFO] uapp: loaded 2584 rows
19:36:06 [INFO] uapp: resolving mutation positions to sequence indices...
19:36:06 [INFO] uapp: position resolution: 2584/2584 mapped (dropped 0)
19:36:06 [INFO] uapp:   train: 1395 mutations
19:36:06 [INFO] uapp:   val: 1019 mutations
19:36:06 [INFO] uapp:   test: 170 mutations
19:36:06 [INFO] uapp: extracting 3Di tokens for 108 unique proteins via foldseek...
foldseek 3Di: 100% 108/108 [00:01<00:00, 59.46prot/s]
19:36:08 [INFO] uapp: 3Di extraction: 100 ok, 8 missing PDB, 0 foldseek failed
19:36:08 [WARNING] uapp: dropping 439 rows whose protein has no 3Di
19:36:08 [INFO] uapp: re-resolving mut_idx against PDB-derived AA strings...
19:36:08 [INFO] uapp: PDB-AA alignment: 2129 ok, 16 unalignable (will be dropped)
19:36:13 [INFO] uapp: loading SaProt: westlake-repl/SaProt_650M_AF2
19:36:14 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/west

## 5. Cache S669 SaProt embeddings (~3 min on T4)

In [11]:
import os, re, pandas as pd

pdb_files = sorted(os.listdir('data/s669/S669/pdbs'))
pdb_set = set(pdb_files)

md = pd.read_csv('cache/s669_metadata.csv')

def find_pdb(code):
    code = str(code).strip()
    # Try several name forms.  S669's pdb_code is e.g. '1a0fA' (4-char PDB id + chain letter).
    # The bundled WT PDB file is named after just the PDB id, e.g. '1a0f.pdb'.
    candidates = []
    candidates.append(f'{code}.pdb')
    candidates.append(f'{code.lower()}.pdb')
    if len(code) >= 4:
        # First 4 chars = PDB id (lowercase or as-is)
        pdb_id = code[:4]
        candidates.append(f'{pdb_id}.pdb')
        candidates.append(f'{pdb_id.lower()}.pdb')
    # Fallback: split on common delimiters
    base = re.split(r'[_:.]', code)[0]
    candidates.append(f'{base}.pdb')
    candidates.append(f'{base.lower()}.pdb')
    for cand in candidates:
        if cand in pdb_set:
            return cand
    return None

md['wt_pdb_basename'] = md['pdb_code'].apply(find_pdb)
n_matched = md['wt_pdb_basename'].notna().sum()
n_unique = md.loc[md['wt_pdb_basename'].notna(), 'pdb_code'].nunique()
print(f"matched {n_matched}/{len(md)} rows  ({n_unique} unique proteins)")
print('first 5 mappings:')
print(md[['pdb_code', 'wt_pdb_basename']].drop_duplicates().head().to_string(index=False))

md_keep = md[md['wt_pdb_basename'].notna()].copy()
md_keep.to_csv('cache/s669_metadata_with_pdb.csv', index=False)
print(f"saved {len(md_keep)} rows")

matched 617/617 rows  (90 unique proteins)
first 5 mappings:
pdb_code wt_pdb_basename
   1a0fA        1a0f.pdb
   1a7vA        1a7v.pdb
   1ba3A        1ba3.pdb
   1bfmA        1bfm.pdb
   1bnlA        1bnl.pdb
saved 617 rows


In [12]:
!python scripts/20_cache_embeddings_saprot.py \
    --metadata-csv cache/s669_metadata_with_pdb.csv \
    --pdb-dir      data/s669/S669/pdbs \
    --pdb-pattern  '{wt_pdb_basename}' \
    --val-fraction 0 --test-fraction 1.0 \
    --out          cache/s669_embeddings_saprot.pt \
    --metadata-out cache/s669_metadata_saprot.csv \
    --device cuda --seed 42

19:42:24 [INFO] uapp: device: cuda
19:42:24 [INFO] uapp: loading metadata: cache/s669_metadata_with_pdb.csv
19:42:24 [INFO] uapp: loaded 617 rows
19:42:24 [INFO] uapp: resolving mutation positions to sequence indices...
19:42:25 [INFO] uapp: position resolution: 617/617 mapped (dropped 0)
19:42:25 [INFO] uapp:   train: 0 mutations
19:42:25 [INFO] uapp:   val: 0 mutations
19:42:25 [INFO] uapp:   test: 617 mutations
19:42:25 [INFO] uapp: extracting 3Di tokens for 90 unique proteins via foldseek...
foldseek 3Di: 100% 90/90 [00:01<00:00, 78.68prot/s]
19:42:26 [INFO] uapp: 3Di extraction: 90 ok, 0 missing PDB, 0 foldseek failed
19:42:26 [INFO] uapp: re-resolving mut_idx against PDB-derived AA strings...
19:42:26 [INFO] uapp: PDB-AA alignment: 615 ok, 2 unalignable (will be dropped)
19:42:32 [INFO] uapp: loading SaProt: westlake-repl/SaProt_650M_AF2
19:42:32 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/westlake-repl/SaProt_650M_AF2/resolve/main/config.json "HTTP/1.1 307 Temporary 

## 6. Build SaProt-aligned bio features

The bio features (k=13) are *backbone-independent* by construction (they depend only on AA identity, RSA, and sequence-derived structural proxies — none of which change when the encoder changes).  We re-build them against the new metadata so row counts align with the SaProt cache, but the standardiser comes from the **T2837 SaProt train split** for T2837 and from the same T2837 mu/sd for S669.

In [13]:
# T2837 SaProt: build standard bio features against the SaProt-aligned metadata
!python scripts/06_build_bio_features.py \
    --metadata-csv cache/t2837_metadata_saprot.csv \
    --embeddings   cache/t2837_embeddings_saprot.pt \
    --out          cache/t2837_bio_features_saprot_extended.pt \
    --include-extended

19:43:10 [INFO] uapp: Embedding-cache split sizes: {'train': 1179, 'val': 813, 'test': 137}
19:43:10 [INFO] uapp: Loaded 2129 metadata rows from cache/t2837_metadata_saprot.csv
19:43:10 [INFO] uapp: CSV split sizes: {'test': 137, 'train': 1179, 'val': 813}
19:43:10 [INFO] uapp:   train: bio-feature tensor shape (1179, 13)
19:43:10 [INFO] uapp:   val: bio-feature tensor shape (813, 13)
19:43:10 [INFO] uapp:   test: bio-feature tensor shape (137, 13)
19:43:10 [INFO] uapp: Saved aligned bio-feature tensor to cache/t2837_bio_features_saprot_extended.pt
19:43:10 [INFO] uapp: Feature names (k=13): ['rsa', 'blosum62', 'grantham', 'delta_charge', 'delta_polarity', 'delta_hydrophobicity', 'delta_volume', 'delta_helix_propensity', 'delta_sheet_propensity', 'local_entropy', 'local_hydrophobic_count', 'local_charged_count', 'position_relative']


In [14]:
# S669 SaProt: inline build standardised with T2837 SaProt train mu/sd
import sys, numpy as np, pandas as pd, torch
sys.path.insert(0, '/content/uapp')
from uapp.data import load_cached_embeddings
from uapp.mutation_features import batch_features_extended

THREE_TO_ONE = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLU':'E','GLN':'Q',
    'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
    'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V',
}
to1 = lambda x: THREE_TO_ONE.get(str(x).strip().upper(), str(x).strip().upper()[:1] or 'X')

md = pd.read_csv('cache/s669_metadata_saprot.csv')
md['split'] = md['split'].astype(str).str.lower()
splits, _ = load_cached_embeddings('cache/s669_embeddings_saprot.pt')
nonempty = [s for s, (X, _) in splits.items() if X.shape[0] > 0]
assert len(nonempty) == 1, nonempty
split_key = nonempty[0]
n_cache = splits[split_key][0].shape[0]
g = md[md['split'] == split_key].reset_index(drop=True)
assert len(g) == n_cache, f'mismatch: md {len(g)} vs cache {n_cache}'

raw = batch_features_extended(
    [to1(x) for x in g['wtAA']], [to1(x) for x in g['mutAA']],
    g['rel_rsa'].astype(float).to_numpy(),
    sequences=g['sequence'].astype(str).tolist(),
    mut_indices=g['mut_idx'].astype(int).to_numpy(),
    include_indicators=False,
)
t_bio = torch.load('cache/t2837_bio_features_saprot_extended.pt',
                   map_location='cpu', weights_only=False)
mu = np.asarray(t_bio['meta']['mu'], dtype=np.float64)
sd = np.asarray(t_bio['meta']['sd'], dtype=np.float64)
sd = np.where(sd < 1e-6, 1.0, sd)
standardised = ((raw - mu) / sd).astype(np.float32)

payload = {k: {'feats': torch.zeros(0, len(mu))} for k in ('train', 'val', 'test')}
payload[split_key] = {'feats': torch.from_numpy(standardised)}
payload['meta'] = {**t_bio['meta'],
                   'source_metadata':   'cache/s669_metadata_saprot.csv',
                   'source_embeddings': 'cache/s669_embeddings_saprot.pt',
                   'standardiser_from': 'cache/t2837_bio_features_saprot_extended.pt'}
torch.save(payload, 'cache/s669_bio_features_saprot_extended.pt')
print(f'Saved cache/s669_bio_features_saprot_extended.pt  ({standardised.shape})')

Saved cache/s669_bio_features_saprot_extended.pt  ((615, 13))


## 7. Train D5 ensemble on T2837 SaProt, evaluate on T2837 test + S669

Same script 18, just different cache files.

In [15]:
!python scripts/18_evaluate_on_s669.py \
    --t2837-emb cache/t2837_embeddings_saprot.pt \
    --t2837-bio cache/t2837_bio_features_saprot_extended.pt \
    --s669-emb  cache/s669_embeddings_saprot.pt \
    --s669-bio  cache/s669_bio_features_saprot_extended.pt \
    --out       outputs/saprot_eval_d5 \
    --ablation D5 --members 5 --device cuda

19:43:32 [INFO] uapp: Device: cuda
19:43:32 [INFO] uapp: Ablation: D5   Members: 5
19:43:32 [INFO] uapp: T2837 bio features: ['rsa', 'blosum62', 'grantham', 'delta_charge', 'delta_polarity', 'delta_hydrophobicity', 'delta_volume', 'delta_helix_propensity', 'delta_sheet_propensity', 'local_entropy', 'local_hydrophobic_count', 'local_charged_count', 'position_relative']
19:43:32 [INFO] uapp: T2837 split sizes: train=1179  val=813  test=137   d_in=2600  d_extra=13
19:43:32 [INFO] uapp: S669 bio features: ['rsa', 'blosum62', 'grantham', 'delta_charge', 'delta_polarity', 'delta_hydrophobicity', 'delta_volume', 'delta_helix_propensity', 'delta_sheet_propensity', 'local_entropy', 'local_hydrophobic_count', 'local_charged_count', 'position_relative']
19:43:32 [INFO] uapp: S669: n=615  (from split 'test')
19:43:32 [INFO] uapp: 
──── Member 1/5 (seed=42) ────
19:43:37 [INFO] uapp: [D5_m0] best val_loss=2.0781
19:43:37 [INFO] uapp: 
──── Member 2/5 (seed=43) ────
19:43:38 [INFO] uapp: [D5_m1] bes

## 8. σ recalibration on S669 (SaProt edition)

Re-fit T on the SaProt predictions — the value is encoder-specific.  Compare to the ESM2 baseline T = 2.76.

In [16]:
!python scripts/19_recalibrate_sigma_s669.py \
    --predictions outputs/saprot_eval_d5/per_member_predictions_s669.npz \
    --out         outputs/saprot_recalibration

19:44:34 [INFO] uapp: Loaded 615 ensemble predictions from outputs/saprot_eval_d5/per_member_predictions_s669.npz
19:44:34 [INFO] uapp: Random split: 308 cal rows, 307 eval rows
19:44:34 [INFO] uapp: Temperature scaling:  T = 2.1008
19:44:34 [INFO] numexpr.utils: NumExpr defaulting to 2 threads.
19:44:34 [INFO] uapp: Isotonic fit on 308 calibration rows.

S669 σ recalibration — eval split (n = 307)
method                         RMSE       NLL       ICE  cov@0.90  cov@0.95   Spearman
----------------------------------------------------------------------------------------------------
baseline_sigma               2.4641    3.0840    0.2709    0.5179    0.6319     0.4395
temperature_scaled           2.4641    2.2686    0.0465    0.9381    0.9902     0.4395
isotonic_calibrated          2.4641    2.2547    0.0567    0.9479    0.9772     0.4618
----------------------------------------------------------------------------------------------------
Temperature T = 2.1008  (σ multiplier).  Spearma

## 9. Save results to Drive + headline comparison

In [17]:
!cp -r outputs/saprot_eval_d5 outputs/saprot_recalibration /content/drive/MyDrive/uapp_cache/
!cp cache/t2837_embeddings_saprot.pt cache/t2837_metadata_saprot.csv \
    cache/t2837_bio_features_saprot_extended.pt \
    cache/s669_embeddings_saprot.pt cache/s669_metadata_saprot.csv \
    cache/s669_bio_features_saprot_extended.pt /content/drive/MyDrive/uapp_cache/
print('saved to Drive')

saved to Drive


In [18]:
import json, pathlib
saprot = json.loads(pathlib.Path('outputs/saprot_eval_d5/ensemble_summary.json').read_text())
saprot_recal = json.loads(pathlib.Path('outputs/saprot_recalibration/recalibration_summary.json').read_text())

# ESM2 reference (from REPORT.md §11)
esm2_t2837   = {'rmse': 1.50, 'nll': 1.85, 'ice': 0.05, 'spearman': 0.348}
esm2_s669    = {'rmse': 2.83, 'nll': 4.64, 'ice': 0.31, 'spearman': 0.434}
esm2_temp_T  = 2.76

saprot_t2837 = saprot['t2837_test']['ensemble']
saprot_s669  = saprot['s669']['ensemble']
saprot_temp_T = saprot_recal['temperature']

row = lambda r: f"  rmse={r['rmse']:.3f}  nll={r['nll']:.3f}  ice={r['ice']:.3f}  spearman={r['spearman']:.3f}"
print('=' * 78)
print('Encoder comparison — ESM2-650M (§11) vs SaProt (this run)')
print('=' * 78)
print('T2837 test (n=170)')
print('  ESM2-650M:', f"  rmse={esm2_t2837['rmse']:.3f}  nll={esm2_t2837['nll']:.3f}  ice={esm2_t2837['ice']:.3f}  spearman={esm2_t2837['spearman']:.3f}")
print('  SaProt:   ', row(saprot_t2837))
print()
print('S669 (n=617)')
print('  ESM2-650M:', f"  rmse={esm2_s669['rmse']:.3f}  nll={esm2_s669['nll']:.3f}  ice={esm2_s669['ice']:.3f}  spearman={esm2_s669['spearman']:.3f}")
print('  SaProt:   ', row(saprot_s669))
print()
print(f'σ recalibration on S669:  ESM2 T = {esm2_temp_T:.2f}    SaProt T = {saprot_temp_T:.2f}')
print('=' * 78)

Encoder comparison — ESM2-650M (§11) vs SaProt (this run)
T2837 test (n=170)
  ESM2-650M:   rmse=1.500  nll=1.850  ice=0.050  spearman=0.348
  SaProt:      rmse=1.598  nll=1.889  ice=0.038  spearman=0.218

S669 (n=617)
  ESM2-650M:   rmse=2.830  nll=4.640  ice=0.310  spearman=0.434
  SaProt:      rmse=2.534  nll=3.185  ice=0.261  spearman=0.416

σ recalibration on S669:  ESM2 T = 2.76    SaProt T = 2.10


## Decision rule

| Outcome | What it means |
|---|---|
| SaProt T2837 RMSE ≤ 1.50 **and** Spearman ≥ 0.348, **and** SaProt S669 RMSE significantly < 2.83 | **Strict win.** Add SaProt as the production encoder; keep D5 head + recalibration recipe. |
| SaProt T2837 unchanged but S669 RMSE unchanged | Structure-awareness doesn't fix the cross-dataset μ gap on its own. Document; try ESM-3 / ESM-IF next. |
| SaProt T2837 *worse* than ESM2 | Encoder swap broke in-distribution performance. Negative result; the gain on S669 (if any) is not free. |
| Any SaProt run with Spearman « 0.348 | The σ-ranking property didn't survive. SaProt's combined AA+3Di tokens may have changed what the σ branch sees. |

## 10. Follow-up A — apples-to-apples ESM2 on the SaProt-aligned 137-row T2837 subset

§12's SaProt T2837 row is on n=137 (the 33 missing test rows belong to 8 proteins whose `uniprot_id` had no AlphaFold-DB model). Comparing it to the §11 ESM2 row on n=170 mixes "encoder change" with "different test population". Script 21 filters the ESM2 cache + bio features to the same 137 rows so the comparison is clean.

In [ ]:
# 10a. Filter ESM2 caches to the SaProt-aligned T2837 subset
!python scripts/21_align_cache_to_reference.py \
    --reference-metadata cache/t2837_metadata_saprot.csv \
    --source-metadata    cache/t2837_metadata.csv \
    --source-emb         cache/t2837_embeddings_v2_650m.pt \
    --source-bio         cache/t2837_bio_features_650m_extended.pt \
    --out-emb            cache/t2837_embeddings_v2_650m_saprot_aligned.pt \
    --out-bio            cache/t2837_bio_features_650m_extended_saprot_aligned.pt

In [ ]:
# 10b. Re-run script 18 with the aligned ESM2 caches.
# This trains a fresh D5 ensemble on T2837 (1179/813 train/val rows — the same
# rows SaProt trained on) and predicts on the same 137-row T2837 test as SaProt.
!python scripts/18_evaluate_on_s669.py \
    --t2837-emb cache/t2837_embeddings_v2_650m_saprot_aligned.pt \
    --t2837-bio cache/t2837_bio_features_650m_extended_saprot_aligned.pt \
    --s669-emb  cache/s669_embeddings_650m.pt \
    --s669-bio  cache/s669_bio_features_650m_extended.pt \
    --out       outputs/esm2_aligned_eval_d5 \
    --ablation D5 --members 5 --device cuda

In [ ]:
# 10c. Side-by-side: ESM2 (137 rows) vs SaProt (137 rows) — the strict apples-to-apples
import json, pathlib

esm2_aligned = json.loads(pathlib.Path('outputs/esm2_aligned_eval_d5/ensemble_summary.json').read_text())
saprot       = json.loads(pathlib.Path('outputs/saprot_eval_d5/ensemble_summary.json').read_text())

row = lambda r: f"  rmse={r['rmse']:.3f}  nll={r['nll']:.3f}  ice={r['ice']:.3f}  spearman={r['spearman']:.3f}"
print('=' * 78)
print('Apples-to-apples — both encoders on the SAME 137-row T2837 test')
print('=' * 78)
print('  ESM2 (aligned):', row(esm2_aligned['t2837_test']['ensemble']))
print('  SaProt:        ', row(saprot['t2837_test']['ensemble']))
print()
print('Same-subset S669 (both ensembles trained on the SaProt-aligned T2837 train):')
print('  ESM2 (aligned):', row(esm2_aligned['s669']['ensemble']))
print('  SaProt:        ', row(saprot['s669']['ensemble']))
print('=' * 78)

!cp -r outputs/esm2_aligned_eval_d5 /content/drive/MyDrive/uapp_cache/
!cp cache/t2837_embeddings_v2_650m_saprot_aligned.pt cache/t2837_bio_features_650m_extended_saprot_aligned.pt /content/drive/MyDrive/uapp_cache/

## 11. Follow-up B — ESM-IF (structure-aware, pure-AA tokens)

§12's hypothesis on the σ-ranking degradation is: SaProt's combined `<aa><3di>` tokens change the input vocabulary, breaking what the σ branch had learned. ESM-IF1 is a structure-aware encoder that reads pure AA tokens and takes structure through GVP geometric features (not the token alphabet). If the σ-ranking on T2837 survives the ESM-IF swap *and* μ on S669 still improves, that's the strict-improvement encoder we were looking for.

Embedding dim is 512 (vs 1280 for ESM2/SaProt), so the mutation-aware feature is 1064-d instead of 2600-d. Downstream FeatureAugmentedHead reads `d_in` dynamically, so scripts 06/18/19 work unchanged.

In [ ]:
# 11a. Install fair-esm (provides esm.inverse_folding for ESM-IF1)
!pip install -q fair-esm
import esm
print('fair-esm version:', esm.__version__ if hasattr(esm, '__version__') else 'installed')

In [ ]:
# 11b. Cache T2837 ESM-IF embeddings
!python scripts/22_cache_embeddings_esmif.py \
    --metadata-csv cache/t2837_metadata.csv \
    --pdb-dir      af_pdbs \
    --pdb-pattern  'AF-{uniprot_id}.pdb' \
    --pdb-chain    'A' \
    --out          cache/t2837_embeddings_esmif.pt \
    --metadata-out cache/t2837_metadata_esmif.csv \
    --device cuda --seed 42

In [ ]:
# 11c. Cache S669 ESM-IF embeddings.
# S669 PDBs may have non-A chains; use the chain_id column from the metadata.
!python scripts/22_cache_embeddings_esmif.py \
    --metadata-csv cache/s669_metadata_with_pdb.csv \
    --pdb-dir      data/s669/S669/pdbs \
    --pdb-pattern  '{wt_pdb_basename}' \
    --pdb-chain-col chain_id \
    --val-fraction 0 --test-fraction 1.0 \
    --out          cache/s669_embeddings_esmif.pt \
    --metadata-out cache/s669_metadata_esmif.csv \
    --device cuda --seed 42

In [ ]:
# 11d. Build ESM-IF-aligned bio features
# T2837: standard script 06 against the ESM-IF processed metadata
!python scripts/06_build_bio_features.py \
    --metadata-csv cache/t2837_metadata_esmif.csv \
    --embeddings   cache/t2837_embeddings_esmif.pt \
    --out          cache/t2837_bio_features_esmif_extended.pt \
    --include-extended

# S669: inline build, standardised with T2837 ESM-IF train mu/sd (same pattern as cell 18)
import sys, numpy as np, pandas as pd, torch
sys.path.insert(0, '/content/uapp')
from uapp.data import load_cached_embeddings
from uapp.mutation_features import batch_features_extended

THREE_TO_ONE = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLU':'E','GLN':'Q',
    'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
    'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V',
}
to1 = lambda x: THREE_TO_ONE.get(str(x).strip().upper(), str(x).strip().upper()[:1] or 'X')

md = pd.read_csv('cache/s669_metadata_esmif.csv')
md['split'] = md['split'].astype(str).str.lower()
splits, _ = load_cached_embeddings('cache/s669_embeddings_esmif.pt')
nonempty = [s for s, (X, _) in splits.items() if X.shape[0] > 0]
assert len(nonempty) == 1, nonempty
split_key = nonempty[0]
n_cache = splits[split_key][0].shape[0]
g = md[md['split'] == split_key].reset_index(drop=True)
assert len(g) == n_cache, f'mismatch: md {len(g)} vs cache {n_cache}'

raw = batch_features_extended(
    [to1(x) for x in g['wtAA']], [to1(x) for x in g['mutAA']],
    g['rel_rsa'].astype(float).to_numpy(),
    sequences=g['sequence'].astype(str).tolist(),
    mut_indices=g['mut_idx'].astype(int).to_numpy(),
    include_indicators=False,
)
t_bio = torch.load('cache/t2837_bio_features_esmif_extended.pt',
                   map_location='cpu', weights_only=False)
mu = np.asarray(t_bio['meta']['mu'], dtype=np.float64)
sd = np.asarray(t_bio['meta']['sd'], dtype=np.float64)
sd = np.where(sd < 1e-6, 1.0, sd)
standardised = ((raw - mu) / sd).astype(np.float32)

payload = {k: {'feats': torch.zeros(0, len(mu))} for k in ('train', 'val', 'test')}
payload[split_key] = {'feats': torch.from_numpy(standardised)}
payload['meta'] = {**t_bio['meta'],
                   'source_metadata':   'cache/s669_metadata_esmif.csv',
                   'source_embeddings': 'cache/s669_embeddings_esmif.pt',
                   'standardiser_from': 'cache/t2837_bio_features_esmif_extended.pt'}
torch.save(payload, 'cache/s669_bio_features_esmif_extended.pt')
print(f'Saved cache/s669_bio_features_esmif_extended.pt  ({standardised.shape})')

In [ ]:
# 11e. Train D5 ensemble on T2837 ESM-IF, evaluate on T2837 test + S669
!python scripts/18_evaluate_on_s669.py \
    --t2837-emb cache/t2837_embeddings_esmif.pt \
    --t2837-bio cache/t2837_bio_features_esmif_extended.pt \
    --s669-emb  cache/s669_embeddings_esmif.pt \
    --s669-bio  cache/s669_bio_features_esmif_extended.pt \
    --out       outputs/esmif_eval_d5 \
    --ablation D5 --members 5 --device cuda

# 11f. σ recalibration on S669 (ESM-IF edition)
!python scripts/19_recalibrate_sigma_s669.py \
    --predictions outputs/esmif_eval_d5/per_member_predictions_s669.npz \
    --out         outputs/esmif_recalibration

# Save to Drive
!cp -r outputs/esmif_eval_d5 outputs/esmif_recalibration /content/drive/MyDrive/uapp_cache/
!cp cache/t2837_embeddings_esmif.pt cache/t2837_metadata_esmif.csv \
    cache/t2837_bio_features_esmif_extended.pt \
    cache/s669_embeddings_esmif.pt cache/s669_metadata_esmif.csv \
    cache/s669_bio_features_esmif_extended.pt /content/drive/MyDrive/uapp_cache/
print('saved to Drive')

In [ ]:
# 11g. Three-way comparison
import json, pathlib

esm2_aligned = json.loads(pathlib.Path('outputs/esm2_aligned_eval_d5/ensemble_summary.json').read_text())
saprot       = json.loads(pathlib.Path('outputs/saprot_eval_d5/ensemble_summary.json').read_text())
esmif        = json.loads(pathlib.Path('outputs/esmif_eval_d5/ensemble_summary.json').read_text())
esmif_recal  = json.loads(pathlib.Path('outputs/esmif_recalibration/recalibration_summary.json').read_text())

row = lambda r: f"  rmse={r['rmse']:.3f}  nll={r['nll']:.3f}  ice={r['ice']:.3f}  spearman={r['spearman']:.3f}"
print('=' * 92)
print('Three-way encoder comparison — ESM2 (aligned) vs SaProt vs ESM-IF')
print('=' * 92)
print('T2837 test')
print('  ESM2 (aligned):', row(esm2_aligned['t2837_test']['ensemble']))
print('  SaProt:        ', row(saprot      ['t2837_test']['ensemble']))
print('  ESM-IF:        ', row(esmif       ['t2837_test']['ensemble']))
print()
print('S669')
print('  ESM2 (aligned):', row(esm2_aligned['s669']['ensemble']))
print('  SaProt:        ', row(saprot      ['s669']['ensemble']))
print('  ESM-IF:        ', row(esmif       ['s669']['ensemble']))
print()
print(f'σ recalibration on S669: ESM-IF T = {esmif_recal["temperature"]:.2f}  '
      f'(ESM2 baseline T = 2.76, SaProt T = 2.10)')
print('=' * 92)
print()
print('Strict-improvement test for ESM-IF:')
e_t = esmif["t2837_test"]["ensemble"]
e_s = esmif["s669"]["ensemble"]
ref_t = esm2_aligned["t2837_test"]["ensemble"]
ref_s = esm2_aligned["s669"]["ensemble"]
print(f"  T2837 RMSE     ESM-IF {e_t['rmse']:.3f} vs ESM2 {ref_t['rmse']:.3f}  "
      f"({'PASS' if e_t['rmse'] <= ref_t['rmse'] else 'FAIL'})")
print(f"  T2837 Spearman ESM-IF {e_t['spearman']:.3f} vs ESM2 {ref_t['spearman']:.3f}  "
      f"({'PASS' if e_t['spearman'] >= ref_t['spearman'] - 0.02 else 'FAIL'})")
print(f"  S669 RMSE      ESM-IF {e_s['rmse']:.3f} vs ESM2 {ref_s['rmse']:.3f}  "
      f"({'PASS' if e_s['rmse'] < ref_s['rmse'] - 0.05 else 'FAIL/NEUTRAL'})")